# Code Interpreter: Python in an OpenAI Sandbox

Use the Responses API to analyse a CSV, generate a chart, and download verified results from a hosted Python sandbox.

[Open in Google Colab](https://colab.research.google.com/github/BrightPool/udemy-prompt-engineering-course/blob/main/openai_features_and_functionality/code_interpreter.ipynb) · [Official guide](https://developers.openai.com/api/docs/guides/tools-code-interpreter)

**The idea:** you describe the analysis; the model writes and runs Python inside a **container** (a sandboxed virtual machine) on OpenAI's infrastructure. Python in *this notebook* only sends the request and downloads the files.

This notebook follows the official guide's **explicit container** pattern: create a container, pass its ID to the `code_interpreter` tool, then download the files the model creates.

You need an OpenAI API key with billing and a model supporting Code Interpreter. Containers are billed at the Code Interpreter built-in tool rates, separately from ChatGPT subscriptions. All data below is synthetic.

In [ ]:
%pip install openai --upgrade --quiet

In [ ]:
# pylint: disable=missing-module-docstring,wrong-import-position
MODEL = "gpt-6.1-sol"
MEMORY_LIMIT = "4g"  # "1g" (default), "4g", "16g" or "64g". Bigger tiers cost more.
CLEANUP = True  # Delete this demo's container and uploaded file after downloading.

In [2]:
import csv
import os
from getpass import getpass
from pathlib import Path
from openai import OpenAI
from IPython.display import display, Markdown, FileLink, Image

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(timeout=180, max_retries=1)
print("Model:", MODEL)

Model: gpt-6.1-sol


## 1. A small dataset we can check ourselves

We want **revenue by region**, a bar chart, and a downloadable summary CSV. Start with six rows so we can verify the answer without trusting the model's prose.

In [3]:
workdir = Path("code_interpreter_demo")
workdir.mkdir(exist_ok=True)
sales_path = workdir / "sales.csv"
sales_path.write_text(
    "month,region,revenue\n"
    "Jan,North,120\nJan,South,100\n"
    "Feb,North,140\nFeb,South,120\n"
    "Mar,North,160\nMar,South,140\n",
    encoding="utf-8",
)
expected_totals = {}
with sales_path.open() as f:
    for row in csv.DictReader(f):
        region = row["region"]
        expected_totals[region] = expected_totals.get(region, 0) + int(row["revenue"])
print(sales_path.read_text())
print("Our check:", expected_totals, "Grand total:", sum(expected_totals.values()))

month,region,revenue
Jan,North,120
Jan,South,100
Feb,North,140
Feb,South,120
Mar,North,160
Mar,South,140

Our check: {'North': 420, 'South': 360} Grand total: 780


## 2. Create a container with the file in it

The guide describes two container modes:

| Mode | How | Use it when |
|---|---|---|
| **Auto** | `"container": {"type": "auto", "memory_limit": "4g", "file_ids": [...]}` | You want OpenAI to create or reuse a container for you |
| **Explicit** | `client.containers.create(...)`, then `"container": container.id` | You want to control the container's files, memory and lifetime |

We use **explicit** mode. We upload the CSV, then create a container with `file_ids` so `sales.csv` is already inside it when the model starts.

A container **expires after 20 minutes without use**. After that, requests that use it fail and its files cannot be recovered.

In [ ]:
with sales_path.open("rb") as f:
    uploaded = client.files.create(file=f, purpose="user_data")

container = client.containers.create(
    name="code-interpreter-demo",
    memory_limit=MEMORY_LIMIT,
    file_ids=[uploaded.id],
)
print("Uploaded file:", uploaded.id)
print("Container:", container.id, container.status)

## 3. Ask the model to use the python tool

Three settings matter here:

- `tools=[{"type": "code_interpreter", "container": container.id}]` gives the model Python in *our* container.
- `tool_choice="required"` makes the model run code instead of doing maths in its head, as in the guide's explicit-container example.
- `include=["code_interpreter_call.outputs"]` returns the execution logs, which the API leaves out by default.

The guide also recommends saying **"use the python tool"** in the prompt.

In [ ]:
response = client.responses.create(
    model=MODEL,
    tools=[{"type": "code_interpreter", "container": container.id}],
    tool_choice="required",
    include=["code_interpreter_call.outputs"],
    instructions=(
        "You are a data analyst. Use the python tool to analyse the supplied CSV "
        "and report results you actually calculated."
    ),
    input=(
        "Use the python tool to read sales.csv. "
        "Calculate total revenue by region and the grand total. "
        "Create regional_totals.csv with exactly the columns region,total_revenue. "
        "Create revenue_by_region.png, a labelled bar chart. "
        "Link both downloadable files in your final answer."
    ),
)
if response.status != "completed":
    raise RuntimeError(
        f"Response stopped with {response.status}; inspect response.incomplete_details."
    )
display(Markdown(response.output_text))

## 4. Inspect what actually ran

An answer is stronger when we can see that the tool executed. Each `code_interpreter_call` item holds the container ID, the Python the model wrote and (because of `include`) its logs.

In [ ]:
calls = [item for item in response.output if item.type == "code_interpreter_call"]
if not calls:
    raise RuntimeError("No Python tool call was returned. Check the model and prompt.")
for call in calls:
    print(f"--- {call.status} in {call.container_id}")
    print(call.code)
    for output in call.outputs or []:
        if output.type == "logs":
            print("Logs:", output.logs)

## 5. Download the generated files and check the numbers

Files the model creates appear as `container_file_citation` annotations on the final message. Each one carries a `container_id`, `file_id` and `filename`. Download them with the **container file content** endpoint (not the normal Files API), and save them before the container expires.

In [ ]:
downloaded = {}
for item in response.output:
    if item.type != "message":
        continue
    for part in item.content:
        for citation in getattr(part, "annotations", []):
            if citation.type != "container_file_citation":
                continue
            # Strip any directory components from the model-provided filename.
            filename = Path(citation.filename).name
            destination = workdir / filename
            content = client.containers.files.content.retrieve(
                citation.file_id, container_id=citation.container_id,
            )
            content.write_to_file(destination)
            downloaded[filename] = destination
            display(FileLink(str(destination)))

summary_path = downloaded.get("regional_totals.csv")
chart_path = downloaded.get("revenue_by_region.png")
if summary_path is None or chart_path is None:
    raise RuntimeError(
        "Expected CSV and PNG were not both cited. "
        "Inspect the output and rerun with clearer filenames."
    )
with summary_path.open() as f:
    actual = {row["region"]: float(row["total_revenue"]) for row in csv.DictReader(f)}
assert actual == expected_totals, f"Calculation mismatch: {actual}"
print("Verified totals:", actual, "Grand total:", sum(actual.values()))
display(Image(filename=str(chart_path)))

## 6. Clean up the remote demo resources

The container would expire on its own after 20 idle minutes, but deleting it is tidier and stops any further use. The downloaded CSV and chart stay in `code_interpreter_demo/`. Colab's local disk is temporary too, so download them to your own computer or Drive if you want to keep them.

In [ ]:
if CLEANUP:
    client.containers.delete(container.id)
    client.files.delete(uploaded.id)
    print("Deleted the demo's container and uploaded file; local results remain.")

## Try it yourself

- Change one revenue value, rerun from the data cell, and check that both the chart and the verified totals change.
- Ask for a monthly trend chart instead of a regional bar chart.
- Switch to **auto** mode: delete the container cell and use `"container": {"type": "auto", "memory_limit": "4g", "file_ids": [uploaded.id]}`. Then find the container the model used in `call.container_id`.

## Summary

- Code Interpreter runs model-written **Python in an OpenAI-hosted container**.
- Create a container (or use auto mode), put your files in it and tell the model to **use the python tool**.
- Add `include=["code_interpreter_call.outputs"]` to see execution logs.
- Download generated files from their `container_file_citation` annotations, and check them against a known answer.
- Containers expire after 20 idle minutes, so save anything you need first.

**Troubleshooting:** update the SDK if `containers` is missing; choose a supported model if the tool is rejected; check API billing if you get a quota error; create a new container if you get an expired-container error. If a cell fails after the container cell, run the cleanup cell once you have downloaded what you need.

Sources: [Code Interpreter guide](https://developers.openai.com/api/docs/guides/tools-code-interpreter), [Container file downloads](https://developers.openai.com/api/reference/resources/containers/subresources/files/subresources/content/methods/retrieve).